---

### What is Function Calling?

`Function calling` allows you to define tools (functions) with a schema (parameters + types), and the LLM will respond by calling that function with a structured JSON payload — not freeform text.

You can then take that structured output and call real Python functions, database queries, APIs, etc.

The tools list can include entities such as:

- `Functions`: Python functions with a defined schema (name, description, parameters).
- `APIs`: Endpoints you want the LLM to call, described with input/output schemas.
- `Database queries`: Functions that interact with databases.
- `File operations`: Tools for reading, writing, or searching files.
- `External services`: Any callable service, like sending emails, fetching weather, or booking tickets.

---

In [2]:
from openai import OpenAI
import json

client = OpenAI()

#### Example : Weather Bot

Ask: "What's the weather in Mumbai?"

LLM calls a `get_weather function` with location info.

`Function Schema`

This tells GPT what function exists and what inputs it expects.

- Top-level: `tools = [ ... ]`

    - This is a list of tool definitions.
    - Each item can be a `function` (like above), or `a file search`, or `other tool types`.
    - You pass this list to `client.chat.completions.create(...)` to tell GPT what functions/tools it can use.

- Inside the List: `{ "type": "function", "function": {...} }`

    - type: "function" tells GPT this is a callable tool.
    - function: Defines the function name, description, and parameters schema — similar to OpenAPI or JSON Schema.

- `function Block`

    ```python
    - "function": {
        "name": "get_weather",
        "description": "Get the current weather in a city",
        ...
    }
    ```
    
    - `name`: What GPT will reference internally. Must be a valid Python identifier (no spaces, special characters).
    - `description`: Human-readable explanation. Helps GPT understand when to call this function.
        - This is critical! The more meaningful and precise this is, the better GPT will match queries to it.
     
- `parameters: JSON Schema to Describe Inputs`

    ```python
    "parameters": {
        "type": "object",
        "properties": { ... },
        "required": ["city"]
    }
    ```
    - You are telling GPT: "This function takes a dictionary (object) as input with specific fields."
    - This is based on the JSON Schema format, which is widely used in API specs.
      
- `properties: Define Each Argument`

    ```python
    "properties": {
        "city": {
            "type": "string",
            "description": "Name of the city to get weather for"
        },
        "units": {
            "type": "string",
            "enum": ["metric", "imperial"],
            "description": "Units for temperature"
        }
    }
    ```
    - city: Required string input — e.g., "Delhi", "New York".
    - units: Optional string input, but values must be "metric" or "imperial".
    - enum: Restricts the valid options (good for closed choices).
        - You can add more enums like "kelvin" if needed.

- `required: List of Required Keys`

    ```python
        "required": ["city"]
    ```

    - Tells GPT that "city" must be included in the arguments.
    - If user says, "Give me the weather", GPT will still try to guess the city if context allows.
    - units is optional. If omitted, your local function should use a default.
    

In [4]:
# Define the function schema as a "tool"
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                    "units": {
                        "type": "string",
                        "enum": ["metric", "imperial"],
                        "default": "metric"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

In [6]:
# User sends a weather query — LLM will suggest a function call
response = client.chat.completions.create(
    model   = "gpt-4o",  # or "gpt-4-turbo"
    messages= [
        {"role": "user", "content": "What’s the weather in Kolkata in Celsius?"}
    ],
    tools       = tools,
    tool_choice = "auto"  # Let model decide when to call
)

In [12]:
# Extract the tool call info from LLM output
# If the LLM decides to call a function, it includes a tool_calls list in the message.
tool_call = response.choices[0].message.tool_calls[0]

# This gets the name of the function the LLM wants to call (e.g., "get_weather").
function_name = tool_call.function.name

# This parses the arguments for the function from a JSON string 
# to a Python dictionary.

# The LLM provides the arguments as a JSON string 
# (e.g., '{"city": "Kolkata", "units": "metric"}').

# json.loads converts this string into a Python dict: 
# {"city": "Kolkata", "units": "metric"}.
function_args = json.loads(tool_call.function.arguments)

print("Function to call:", function_name)
print("Arguments:", function_args)

Function to call: get_weather
Arguments: {'city': 'Kolkata', 'units': 'metric'}


`Function to Implement`

In [7]:
def get_weather(city, units="metric"):
    # Simulate API call
    if city.lower() == "delhi":
        return {
            "temperature": "34°C",
            "condition": "Sunny",
            "humidity": "45%",
            "units": units
        }
        
    return {
        "temperature": "Unknown",
        "condition": "Unknown",
        "units": units
    }

In [13]:
def get_weather(city, units="metric"):
    # Stub implementation
    return f"The weather in {city} is 34°C and sunny (units: {units})"

In [14]:
# Actually call the function and get result
function_result = get_weather(**function_args)

print("Function Result:", function_result)

Function Result: The weather in Kolkata is 34°C and sunny (units: metric)


In [9]:
# Send result back to LLM to generate natural reply
# after we’ve run the function
followup_response = client.chat.completions.create(
    model   ="gpt-4o",
    messages=[                                     # conversation history we send to the LLM.
            # original user question.
            {"role": "user", "content": "What’s the weather in Kolkata in Celsius?"},
            # The assistant’s tool call (what function it decided to call and with what arguments)
            # The LLM’s tool call (echoed back to the LLM)
            # The LLM expects to see its previous decision (the tool call) 
            # before you provide the tool’s result.
            # This lets the LLM link the function output to the correct tool call, 
            # ensuring it generates a relevant, accurate natural language reply.
            # The "role": "assistant" is used to indicate that 
            # this message comes from the AI assistant (the LLM)
            # essential for the LLM to track its own decisions and 
            # maintain context in multi-step interactions.
            {
                "role": "assistant",
                "tool_calls": [                    # Echo what the assistant (LLM) just decided earlier
                    {
                        "id": tool_call.id,        # tool_call.id from earlier tool call
                        "type": "function",
                        "function": {
                            "name": function_name,
                            "arguments": json.dumps(function_args)   # e.g. {"city": "Kolkata", "units": "metric"}
                        }
                    }
                ]
            },
            # The tool result (the output of your Python function).
            # send the result of the Python function back to the LLM
            {
                # Indicates that this message is from a tool (your Python function), 
                # not the user or assistant.
                "role": "tool",                  # Now you as developer inject the function result here
                "tool_call_id": tool_call.id,    # Match the ID
                "content": function_result       # Output of get_weather()
            }
    ]
)

# LLM uses the function result to generate a final, natural language reply for the user. 
print("Final Answer:", followup_response.choices[0].message.content)

Final Answer: The current weather in Kolkata is 34°C and sunny.


**Why do we need the above call?**
    
The initial response doesn't give you the final answer. 

It only says:

> "To answer this, I need to call a function named `get_weather with arguments...`"

After you (as the developer) execute that function (Cell 6), the model needs to see that output before it can give the final reply.

**tool loop:**

- User asks something.
- Assistant (LLM) says: “I’ll call function X with args Y”.
- You (the developer) run that function.
- You tell the LLM: “Here’s the result of that function”.
- LLM then produces the final natural language answer.

```text
User: What’s the weather in Kolkata in Celsius?

Assistant: [calls get_weather(city=Kolkata, units=metric)]

You run the function and return:
→ "The weather in Kolkata is 34°C and sunny"

Assistant: "The weather in Kolkata is 34°C and sunny."
```

**The 2 LLM Calls**

🔹 Call 1: Detect the Need for a Tool

- What happens?
    - LLM does not return a direct answer.
    - Instead, it returns:
        - “I want to call the function get_weather with args {"city": "Kolkata", "units": "metric"}”.
    - This is often referred to as a `tool call` or `function invocation`.

🔹 Call 2: Final Answer After Tool Execution

This separation allows trust, security, and control — you (developer) own the tool execution, not the model.

**Agentic Loop Flow**

```python
[Step 1] LLM: Reasoning → selects tool & arguments
        ↓
[Step 2] Agent: Executes tool with args
        ↓
[Step 3] Agent: Updates context/memory
        ↓
[Step 4] LLM: Next reasoning step → choose next action
        ↓
... repeat until done
```

| Task                        | Plain Function Calling | Agentic AI (e.g. LangGraph) |
| --------------------------- | ---------------------- | --------------------------- |
| Tool decision by LLM        | ✅                     | ✅                         |
| Tool execution by developer | (manual)                |  (automatic by agent)      |
| Memory across steps         | (you manage it)         | ✅ (built-in memory)      |
| Multi-step workflows        | Manual chaining         | Autonomous loop            |


---
#### Travel Planning Assistant with Function Calling + Pydantic
---

To automate personalized travel itinerary generation using a conversational assistant that:

- Understands natural language user inputs (e.g., `Plan a trip to Manali in October for 12 days`)
- Extracts structured travel parameters (destination, duration, budget, month)
- Uses function calling to trigger a backend planner
- Returns a user-friendly, tailored travel plan

In [15]:
from openai import OpenAI
from pydantic import BaseModel, Field, field_validator, ValidationError
import json
from pprint import pprint

client = OpenAI()

#### Define Pydantic Model (TripRequest)

In [29]:
# class TripRequest(BaseModel):
#     destination: str   = Field(...,                      description="Travel destination")
#     duration_days: int = Field(..., ge=1, le=60,         description="Trip duration (1–60 days)")
#     budget_inr: int    = Field(..., ge=10000, le=500000, description="Budget in INR")
#     month: str         = Field(...,                      description="Month name")

#     @field_validator("month")
#     def validate_month(cls, v):
#         months = [
#             "january", "february", "march",     "april",   "may",      "june",
#             "july",    "august",   "september", "october", "november", "december"
#         ]
#         if v.lower() not in months:
#             raise ValueError("Month must be a valid name like 'October'")
            
#         return v.capitalize()

#### Define Function/tool Schema (OpenAI tool format)

In [16]:
tools = [
    {
        "type": "function",                           # Required type
        "function": {
            "name": "plan_trip",                      # Name of your function
            "description": "Plan an itinerary for a user based on location, dates, and budget",
            "parameters": {
                "type": "object",                     # Top-level schema type
                "properties": {
                    "destination": {
                        "type": "string",
                        "description": "Target destination country or city"
                    },
                    "duration_days": {
                        "type": "integer",
                        "description": "Trip duration in days"
                    },
                    "budget_inr": {
                        "type": "integer",
                        "description": "Total budget in INR"
                    },
                    "month": {
                        "type": "string",
                        "description": "Month of the trip"
                    }
                },
                "required": ["destination", "duration_days", "budget_inr", "month"]
            }
        }
    }
]

| Field                | Explanation                                                        |
| -------------------- | ------------------------------------------------------------------ |
| `"type": "function"` | Informs OpenAI that this is a function tool                        |
| `"name"`             | Name of the function the LLM can "call"                            |
| `"description"`      | Describes what the function does (helps LLM choose the right tool) |
| `"parameters"`       | JSON Schema to validate input arguments                            |
| `"type": "object"`   | Indicates the function accepts a dictionary-like input             |
| `"properties"`       | Defines each parameter (name, type, and description)               |
| `"required"`         | Specifies which fields are mandatory                               |


#### Define a Pydantic Model for Validated Use

In [17]:
from pydantic import BaseModel, Field, validator, field_validator

In [18]:
class TripRequest(BaseModel):
    destination: str   = Field(...,                      description="Travel destination")
    duration_days: int = Field(..., ge=1, le=60,         description="Trip duration (1–60 days)")
    budget_inr: int    = Field(..., ge=10000, le=500000, description="Budget in INR")
    month: str         = Field(...,                      description="Month name")

    @field_validator("month")
    def validate_month(cls, v):
        months = [
            "january", "february", "march",     "april",   "may",      "june",
            "july",    "august",   "september", "october", "november", "december"
        ]
        
        if v.lower() not in months:
            raise ValueError("Month must be a valid name like 'October'")
            
        return v.capitalize()

#### LLM Call – Simulate User Query

In [19]:
response = client.chat.completions.create(
    model      = "gpt-4-1106-preview",
    messages   = [
        {"role": "user", 
         "content": "I want to travel to Manali for 5 days in October, and my budget is 40000 INR"}
    ],
    tools      = tools,
    tool_choice= "auto"
)

#### Extract Tool Call Arguments & Validate

In [20]:
tool_call = response.choices[0].message.tool_calls[0]

print("Function to call:", tool_call.function.name)
print("Arguments:", tool_call.function.arguments)

parsed_args = json.loads(tool_call.function.arguments)

try:
    trip = TripRequest(**parsed_args)
    
    print("\n✅ Validated Trip Request:")
    pprint(trip.model_dump(), indent=2)
    
except ValidationError as e:
    print("❌ Validation failed:")
    print(e)

Function to call: plan_trip
Arguments: {
  "destination": "Manali",
  "duration_days": 5,
  "budget_inr": 40000,
  "month": "October"
}

✅ Validated Trip Request:
{ 'budget_inr': 40000,
  'destination': 'Manali',
  'duration_days': 5,
  'month': 'October'}


#### Define and Run Tool (Plan Trip)

In [21]:
def plan_trip(destination, duration_days, budget_inr, month):
    return f"""
        ✅ Here's your travel plan to **{destination}**:

        - Duration: {duration_days} days in {month}
        - Budget: ₹{budget_inr:,}
        - Recommended: Book early for scenic views, explore cafes, try local treks.
        - {destination} in {month} is usually a great time to visit!
    """

In [22]:
result = plan_trip(**trip.model_dump())
print(result)


        ✅ Here's your travel plan to **Manali**:

        - Duration: 5 days in October
        - Budget: ₹40,000
        - Recommended: Book early for scenic views, explore cafes, try local treks.
        - Manali in October is usually a great time to visit!
    


| Step                   | Description                                                                     |
| ---------------------- | ------------------------------------------------------------------------------- |
| 🔧 Tool Schema         | Defined a `plan_trip` function schema using OpenAI's tool format                |
| 🧠 LLM Prompt          | Prompted the model with a user query (e.g., “Plan a trip to Manali in October”) |
| 🧾 LLM Output          | Model selected the function `plan_trip` and filled in valid arguments           |
| 🛠️ Function Execution | We **manually** invoked the Python function using the arguments                 |
| 💬 Final Output        | The function echoed back the inputs in a nicely formatted string                |


- This was just a `function-calling demonstration`: showing how the LLM can reason which function to call and what arguments to pass.
- But the actual function (`plan_trip`) was just a `placeholder` — no real computation, intelligence, or lookup happened.

---
#### Multi-turn conversation using memory via Threads API (in OpenAI's Responses mode)

---
Goal : Make it a multi-turn flow like:

- `User Turn 1`: I want to plan a trip to Manali in October.
- `Assistant`: Asks for missing info (e.g., budget, duration).
- `User Turn 2`: 5 days, ₹40,000
- `Assistant`: Completes the plan using memory of earlier inputs.

**1. Define the tool and function schema (same as before)**

In [83]:
tools = [
    {
        "type": "function",                           # Required type
        "function": {
            "name": "plan_trip",                      # Name of your function
            "description": "Plan an itinerary for a user based on location, dates, and budget",
            "parameters": {
                "type": "object",                     # Top-level schema type
                "properties": {
                    "destination": {
                        "type": "string",
                        "description": "Target destination country or city"
                    },
                    "duration_days": {
                        "type": "integer",
                        "description": "Trip duration in days"
                    },
                    "budget_inr": {
                        "type": "integer",
                        "description": "Total budget in INR"
                    },
                    "month": {
                        "type": "string",
                        "description": "Month of the trip"
                    }
                },
                "required": ["destination", "duration_days", "budget_inr", "month"]
            }
        }
    }
]

**2: Implement the trip planner function**

In [84]:
def plan_trip(destination, duration_days, budget_inr, month):
    return f"""
        Trip Plan for **{destination}**:
        - Month: {month}
        - Duration: {duration_days} days
        - Budget: ₹{budget_inr:,}
        - Tips: Pack light, check weather updates, pre-book stays.
    """

**3: Set up thread and add memory**

In [85]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [86]:
client = OpenAI()

In [87]:
# Create a new thread (used to maintain memory)
thread = client.beta.threads.create()

# OpenAI recommends transitioning to the newer client.chat.completions.create() with tools 
# and threads, which is part of the Responses API (as of July 2024).

**4: Add message (Turn 1)**

In [88]:
# Turn 1 message: Only partial info
client.beta.threads.messages.create(
    thread_id = thread.id,
    role      = "user",
    content   = "I want to visit Delhi in October. Can you plan a trip?"
)

# content   = "I want to visit Manali in October for 6 days and I have ₹30000 to spend. Can you plan a trip?"

Message(id='msg_8SCY79VKVR7HRYBdRKr8qVNU', assistant_id=None, attachments=[], completed_at=None, content=[TextContentBlock(text=Text(annotations=[], value='I want to visit Delhi in October. Can you plan a trip?'), type='text')], created_at=1757913205, incomplete_at=None, incomplete_details=None, metadata={}, object='thread.message', role='user', run_id=None, status=None, thread_id='thread_lcFiqCQ40NZaytfs0T6F81m3')

In [89]:
messages = client.beta.threads.messages.list(thread_id=thread.id)
for m in messages.data:
    print(f"{m.role}: {m.content[0].text.value}")

user: I want to visit Delhi in October. Can you plan a trip?


**5: Run assistant with tool**

In [90]:
assistant = client.beta.assistants.create(
    name        = "Travel Planner Assistant",
    instructions= "You are a travel assistant that helps users plan trips using the provided tool.",
    tools       = tools,  # `plan_trip` function schema
    model       = "gpt-4-turbo"
)

In [91]:
messages = client.beta.threads.messages.list(thread_id=thread.id)
for m in messages.data:
    print(f"{m.role}: {m.content[0].text.value}")

user: I want to visit Delhi in October. Can you plan a trip?


In [92]:
print(assistant.tools)  # Should not be empty

[FunctionTool(function=FunctionDefinition(name='plan_trip', description='Plan an itinerary for a user based on location, dates, and budget', parameters={'type': 'object', 'properties': {'destination': {'type': 'string', 'description': 'Target destination country or city'}, 'duration_days': {'type': 'integer', 'description': 'Trip duration in days'}, 'budget_inr': {'type': 'integer', 'description': 'Total budget in INR'}, 'month': {'type': 'string', 'description': 'Month of the trip'}}, 'required': ['destination', 'duration_days', 'budget_inr', 'month']}, strict=False), type='function')]


In [93]:
run = client.beta.threads.runs.create_and_poll(
    thread_id   = thread.id,
    assistant_id= assistant.id,   # Your assistant with `plan_trip` as tool
    tools       = tools,
    tool_choice = "auto"
)

# This step internally invokes the LLM, matches the tool, extracts arguments, 
# and prepares a tool call.

# Defining tools at the Assistant level sets the default tools for all interactions.
# Passing tools in the run call lets you customize or extend the toolset for a 
# particular thread or run.
# If you don’t pass tools in the run, the assistant’s default tools are used.

In [94]:
messages = client.beta.threads.messages.list(thread_id=thread.id)
for m in messages.data:
    print(f"{m.role}: {m.content[0].text.value}")

assistant: Sure, I can help with that! Could you please provide me with the duration of your trip and your budget in INR? This will help me plan the best possible itinerary for you.
user: I want to visit Delhi in October. Can you plan a trip?


_Debug_

In [95]:
# Gets all messages in the thread.
messages = client.beta.threads.messages.list(thread_id=thread.id)

# Loops through the messages in reverse order (from newest to oldest).
for m in messages.data[::-1]:
    
    # Finds the first message where the role is "assistant" (i.e., a reply from the LLM).
    if m.role == "assistant":
        print(m.content[0].text.value)
        break

Sure, I can help with that! Could you please provide me with the duration of your trip and your budget in INR? This will help me plan the best possible itinerary for you.


**How/When does this happen?**

- You send a user message like “I want to visit Delhi in October.”
- The LLM checks the tool schema and sees that duration_days and budget_inr are required.
- Instead of making a `tool call`, the LLM replies as an assistant, prompting the user for the missing info.

**Summary:**

The LLM automatically asks for missing required parameters when the user’s input is incomplete, based on the tool’s schema.

the LLM is `intelligent` in the sense that it can understand context, follow instructions, identify missing information, and interact conversationally.

`Thread-based memory is working`

The assistant remembered your earlier query about a travel destination and is asking just for the missing details (duration and budget). This is the essence of multi-turn conversations with memory.

**6: Extract Tool Call Arguments from the LLM**

- check if the LLM has requested a tool call in the current run


In [96]:
# it means the LLM wants you to execute a tool (function).
if run.required_action and run.required_action.submit_tool_outputs:
    tool_call = run.required_action.submit_tool_outputs.tool_calls[0]
    
    print("Tool name:", tool_call.function.name)
    print("Arguments:", tool_call.function.arguments)
else:                # If not, it prints “No tool call yet. Assistant may need more info.”
    print("No tool call yet. Assistant may need more info.")

No tool call yet. Assistant may need more info.


In [97]:
# tool_call = run.required_action.submit_tool_outputs.tool_calls[0]

# print("Tool name:", tool_call.function.name)
# print("Arguments:", tool_call.function.arguments)

# if run.required_action and run.required_action.submit_tool_outputs:
#     tool_call = run.required_action.submit_tool_outputs.tool_calls[0]
    
#     print("Tool name:", tool_call.function.name)
#     print("Arguments:", tool_call.function.arguments)
# else:
#     print("No tool call detected for this run.")

**Turn 2**

In [98]:
# sends a new message to the thread, representing the user's second turn in the conversation. 
# The user provides the missing details (budget and duration) that the assistant previously requested.
client.beta.threads.messages.create(
    thread_id = thread.id,
    role      = "user",
    content   = "I can spend ₹40,000 and stay for 5 days."
)

Message(id='msg_226b9gGdUYuD9plQZX6Sn9Yd', assistant_id=None, attachments=[], completed_at=None, content=[TextContentBlock(text=Text(annotations=[], value='I can spend ₹40,000 and stay for 5 days.'), type='text')], created_at=1757913223, incomplete_at=None, incomplete_details=None, metadata={}, object='thread.message', role='user', run_id=None, status=None, thread_id='thread_lcFiqCQ40NZaytfs0T6F81m3')

In [99]:
messages = client.beta.threads.messages.list(thread_id=thread.id)
for m in messages.data:
    print(f"{m.role}: {m.content[0].text.value}")

user: I can spend ₹40,000 and stay for 5 days.
assistant: Sure, I can help with that! Could you please provide me with the duration of your trip and your budget in INR? This will help me plan the best possible itinerary for you.
user: I want to visit Delhi in October. Can you plan a trip?


In [100]:
# code starts a new run in the thread, 
# asking the assistant (LLM) to process the latest user message and decide what to do next.
run = client.beta.threads.runs.create_and_poll(
    thread_id   = thread.id,      # which conversation thread to use.
    assistant_id= assistant.id,   # Your assistant with `plan_trip` as tool
    tools       = tools,          # Provides the list of available tools/functions for this run.
    tool_choice = "auto"          # Allows the LLM to decide whether to call a tool or respond directly.
)

- When you start a run, the assistant (LLM) processes the entire conversation history in the thread, not just the latest message. 
- However, the latest user message is what typically triggers the next action, but the LLM uses all previous context to make its decision. 
- So, it considers everything in the thread, but responds based on the most recent user input and the accumulated context.

In [101]:
messages = client.beta.threads.messages.list(thread_id=thread.id)
for m in messages.data:
    print(f"{m.role}: {m.content[0].text.value}")

user: I can spend ₹40,000 and stay for 5 days.
assistant: Sure, I can help with that! Could you please provide me with the duration of your trip and your budget in INR? This will help me plan the best possible itinerary for you.
user: I want to visit Delhi in October. Can you plan a trip?


In [102]:
if run.required_action and run.required_action.submit_tool_outputs:
    tool_call = run.required_action.submit_tool_outputs.tool_calls[0]
    
    print("Tool name:", tool_call.function.name)
    print("Arguments:", tool_call.function.arguments)
else:
    print("No tool call yet. Assistant may need more info.")

Tool name: plan_trip
Arguments: {"destination":"Delhi","duration_days":5,"budget_inr":40000,"month":"October"}


**Memory in Action (Thread-based)**

`Turn 1: Partial Input`

> "I want to visit Delhi in October."

    - LLM does not have all four required fields (destination, duration_days, budget_inr, month)
    - So it doesn't trigger your plan_trip function.

`Turn 2: Completion Info`

> "I can spend ₹40000 and stay for 5 days."
> 
    - You didn’t repeat destination or month — but that’s fine!
    - The LLM uses thread memory to combine inputs across turns.

`Result:`

```python
Tool name: plan_trip
Arguments: {"destination":"Delhi","duration_days":5,"budget_inr":40000,"month":"October"}
```

**call the function**

In [103]:
# Extract the tool call from the previous run
tool_call = run.required_action.submit_tool_outputs.tool_calls[0]
print("Tool name:", tool_call.function.name)

Tool name: plan_trip


In [104]:
# Get function name and arguments
function_name = tool_call.function.name
arguments     = json.loads(tool_call.function.arguments)

print("Arguments:", tool_call.function.arguments)

Arguments: {"destination":"Delhi","duration_days":5,"budget_inr":40000,"month":"October"}


In [105]:
# Run the actual function
result = plan_trip(**arguments)
print(result)


        Trip Plan for **Delhi**:
        - Month: October
        - Duration: 5 days
        - Budget: ₹40,000
        - Tips: Pack light, check weather updates, pre-book stays.
    


In [106]:
run_status = client.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)
print("Current run status:", run_status.status)  # Debugging

Current run status: requires_action


STOP AT THIS POINT

---
#### Now test a follow-up message (multi-turn)
---

**Turn 3**

`Purpose:`

It allows the assistant (LLM) to update its context and respond with a revised plan or next steps, demonstrating multi-turn, stateful conversation.

In [172]:
# pass the tool output back to the LLM to generate a final response.
# This step tells the OpenAI system that you have executed the function
    
client.beta.threads.runs.submit_tool_outputs(
    thread_id= thread.id,
    run_id   = run.id,
    tool_outputs=[
        {
            "tool_call_id": tool_call.id,
            "output": result
        }
    ]
)

Run(id='run_Z8jLYaCxlXj8SvGbMg0B5I73', assistant_id='asst_lxBIijbfJBU8TgZiCNuQVYkQ', cancelled_at=None, completed_at=None, created_at=1757617640, expires_at=1757618240, failed_at=None, incomplete_details=None, instructions='You are a travel assistant that helps users plan trips using the provided tool.', last_error=None, max_completion_tokens=None, max_prompt_tokens=None, metadata={}, model='gpt-4-turbo', object='thread.run', parallel_tool_calls=True, required_action=None, response_format='auto', started_at=1757617640, status='queued', thread_id='thread_je6HyAgYSmT9IvzpC5yBYWqZ', tool_choice='auto', tools=[FunctionTool(function=FunctionDefinition(name='plan_trip', description='Plan an itinerary for a user based on location, dates, and budget', parameters={'type': 'object', 'properties': {'destination': {'type': 'string', 'description': 'Target destination country or city'}, 'duration_days': {'type': 'integer', 'description': 'Trip duration in days'}, 'budget_inr': {'type': 'integer', '

In [173]:
run_status = client.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)
print("Current run status:", run_status.status)  # Debugging

Current run status: completed


In [174]:
messages = client.beta.threads.messages.list(thread_id=thread.id)
for m in messages.data:
    print(f"{m.role}: {m.content[0].text.value}")

assistant: Great! Here’s your trip plan for Delhi in October:

- **Duration**: 5 days
- **Budget**: ₹40,000
- **Tips**: It's a good idea to pack light and check the weather updates as it can vary. Also, remember to pre-book your stays to avoid any last-minute hassles.

If you need more specific recommendations on places to visit, where to stay, or any other details, feel free to ask! Enjoy your trip to Delhi!
user: I can spend ₹40,000 and stay for 5 days.
assistant: Sure! I can help you plan your trip to Delhi. Could you please provide me with the following details:

1. How many days do you plan to stay in Delhi?
2. What is your budget for the trip in Indian Rupees (INR)?
user: I want to visit Delhi in October. Can you plan a trip?


In [142]:
# import time

# # Poll until the run is completed
# while True:
#     run_status = client.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)
#     if run_status.status in ["completed", "failed", "cancelled", "expired"]:
#         break
#     time.sleep(1)  # Wait 1 second before checking again

new message from user:

In [175]:
# User follow-up: Changes duration and budget
client.beta.threads.messages.create(
    thread_id = thread.id,
    role      = "user",
    content   = "Actually, I want to make it a 7-day trip with ₹60,000 budget."
)

Message(id='msg_w9iLP8tZSNOQRfYYm1AlYSl8', assistant_id=None, attachments=[], completed_at=None, content=[TextContentBlock(text=Text(annotations=[], value='Actually, I want to make it a 7-day trip with ₹60,000 budget.'), type='text')], created_at=1757617902, incomplete_at=None, incomplete_details=None, metadata={}, object='thread.message', role='user', run_id=None, status=None, thread_id='thread_je6HyAgYSmT9IvzpC5yBYWqZ')

In [176]:
messages = client.beta.threads.messages.list(thread_id=thread.id)
for m in messages.data:
    print(f"{m.role}: {m.content[0].text.value}")

user: Actually, I want to make it a 7-day trip with ₹60,000 budget.
assistant: Great! Here’s your trip plan for Delhi in October:

- **Duration**: 5 days
- **Budget**: ₹40,000
- **Tips**: It's a good idea to pack light and check the weather updates as it can vary. Also, remember to pre-book your stays to avoid any last-minute hassles.

If you need more specific recommendations on places to visit, where to stay, or any other details, feel free to ask! Enjoy your trip to Delhi!
user: I can spend ₹40,000 and stay for 5 days.
assistant: Sure! I can help you plan your trip to Delhi. Could you please provide me with the following details:

1. How many days do you plan to stay in Delhi?
2. What is your budget for the trip in Indian Rupees (INR)?
user: I want to visit Delhi in October. Can you plan a trip?


In [177]:
run_status = client.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)
print("Current run status:", run_status.status)  # Debugging

Current run status: completed


In [178]:
run = client.beta.threads.runs.create_and_poll(
    thread_id   = thread.id,
    assistant_id= assistant.id,
    tool_choice = "auto"
)

In [179]:
messages = client.beta.threads.messages.list(thread_id=thread.id)
for m in messages.data:
    print(f"{m.role}: {m.content[0].text.value}")

user: Actually, I want to make it a 7-day trip with ₹60,000 budget.
assistant: Great! Here’s your trip plan for Delhi in October:

- **Duration**: 5 days
- **Budget**: ₹40,000
- **Tips**: It's a good idea to pack light and check the weather updates as it can vary. Also, remember to pre-book your stays to avoid any last-minute hassles.

If you need more specific recommendations on places to visit, where to stay, or any other details, feel free to ask! Enjoy your trip to Delhi!
user: I can spend ₹40,000 and stay for 5 days.
assistant: Sure! I can help you plan your trip to Delhi. Could you please provide me with the following details:

1. How many days do you plan to stay in Delhi?
2. What is your budget for the trip in Indian Rupees (INR)?
user: I want to visit Delhi in October. Can you plan a trip?


still the same messgage stack

becasue..

In [180]:
if run.required_action and run.required_action.submit_tool_outputs:
    tool_call = run.required_action.submit_tool_outputs.tool_calls[0]
    
    print("Tool name:", tool_call.function.name)
    print("Arguments:", tool_call.function.arguments)
else:
    print("No tool call yet. Assistant may need more info.")

Tool name: plan_trip
Arguments: {"destination":"Delhi","duration_days":7,"budget_inr":60000,"month":"October"}


In [181]:
run_status = client.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)
print("Current run status:", run_status.status)  # Debugging

Current run status: requires_action


> `Multi-turn` is less about “better LLMs” and more about thoughtful dialogue design + structured memory + intent orchestration.

---
#### Patient care use case

---

| Layer                 | What we’ll do                                                   |
| --------------------- | --------------------------------------------------------------- |
| **Use case**          | Capture structured **Patient Info** from natural language input |
| **Pydantic model**    | Define a schema for patient fields (age, gender, symptoms...)   |
| **Function calling**  | Use OpenAI function calling to extract structured values        |
| **Response API**      | Use the newer `client.chat.completions.create()` flow           |
| **Memory (optional)** | Handle follow-ups via multi-turn conversation logic             |

**Example User Query:**

"Hi, I have a 65-year-old male patient with chronic cough, occasional chest pain, and fatigue. Could you assist with next steps?"

**1. Define the Pydantic Model**

In [107]:
from pydantic import BaseModel, Field, field_validator
from typing import Literal, Optional

In [108]:
class PatientInfo(BaseModel):
    name: Optional[str]                        = Field(None,               description="Full name of the patient")
    age: int                                   = Field(..., ge=0, le=120,  description="Age in years")
    gender: Literal["male", "female", "other"] = Field(...,                description="Gender of the patient")
    symptoms: list[str]                        = Field(...,                description="List of reported symptoms")
    duration_days: Optional[int]               = Field(None, ge=1, le=365, description="Duration of illness in days")

    @field_validator("symptoms")
    @classmethod
    def validate_symptoms(cls, v):
        if not isinstance(v, list) or not all(isinstance(i, str) for i in v):
            raise ValueError("Symptoms must be a list of strings.")
        return v

**2: Define Tool Schema**

In [109]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "register_patient",
            "description": "Extract structured patient info from unstructured input",
            "parameters": PatientInfo.model_json_schema()
        }
    }
]

**3. LLM Call using Response API**

In [110]:
from openai import OpenAI

In [111]:
client = OpenAI()

In [112]:
response = client.chat.completions.create(
    model="gpt-4-1106-preview",
    messages=[
        {
            "role": "system",
            "content": "You are a medical assistant that extracts structured patient information."
        },
        {
            "role": "user",
            "content": "Patient is a 65-year-old male, has chronic cough and chest pain for a week."
        }
    ],
    tools      = tools,
    tool_choice= "auto"
)

**4: Extract Function & Arguments**

In [114]:
tool_call = response.choices[0].message.tool_calls[0]

print("Function to call:", tool_call.function.name)
print("Arguments:", tool_call.function.arguments)

Function to call: register_patient
Arguments: {
  "name": null,
  "age": 65,
  "gender": "male",
  "symptoms": ["chronic cough", "chest pain"],
  "duration_days": 7
}


**5: Validate & Execute Locally**

In [115]:
import json

args = json.loads(tool_call.function.arguments)

try:
    patient = PatientInfo(**args)
    print("✅ Patient registered:", patient)
except Exception as e:
    print("❌ Validation failed:", e)

✅ Patient registered: name=None age=65 gender='male' symptoms=['chronic cough', 'chest pain'] duration_days=7


---
#### Use Case: Registering a Patient with Incomplete Info (Multi-Turn)
---

Handle a patient registration scenario where the user initially provides partial info, and the assistant helps extract the rest over follow-up turns.

In [134]:
from openai import OpenAI
import json

In [135]:
client = OpenAI()

**1: Define the PatientInfo Pydantic Model**

In [136]:
from pydantic import BaseModel, Field, field_validator
from typing import Literal, Optional

In [137]:
class PatientInfo(BaseModel):
    name: Optional[str]                         = Field(None, description="Patient's full name")
    age: int                                    = Field(..., ge=0, le=120, description="Age in years")
    gender: Literal["male", "female", "other"]  = Field(..., description="Gender")
    symptoms: list[str]                         = Field(..., description="List of symptoms")
    duration_days: Optional[int]                = Field(None, ge=1, le=365, description="How long symptoms have been present")

    @field_validator("symptoms")
    @classmethod
    def validate_symptoms(cls, v):
        if not isinstance(v, list):
            raise ValueError("Symptoms must be a list of strings.")
        return v

**2: Define Tool Schema for Function Calling**

In [138]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "register_patient",
            "description": "Capture structured patient info from natural language",
            "parameters": PatientInfo.model_json_schema()
        }
    }
]

**3: Simulate Multi-Turn Using Response API**

- Turn 1: Partial Information

In [139]:
# First user message (partial)
messages = [
    {
        "role": "system",
        "content": "You are a medical assistant that captures patient records."
    },
    {
        "role": "user",
        "content": "I have a 70-year-old female patient with headache."
    }
]

In [140]:
response1 = client.chat.completions.create(
    model      = "gpt-4-1106-preview",
    messages   = messages,
    tools      = tools,
    tool_choice= "auto"
)

In [141]:
tool_call1 = response1.choices[0].message.tool_calls[0]
args1      = json.loads(tool_call1.function.arguments)

print("Function Arguments from Turn 1:")
print(args1)

Function Arguments from Turn 1:
{'name': None, 'age': 70, 'gender': 'female', 'symptoms': ['headache'], 'duration_days': None}


- Turn 2: User Adds More Info

In [142]:
# Append new message (user adds more info)
messages.append({
    "role": "user",
    "content": "She also has fever and nausea since 3 days."
})

In [143]:
response2 = client.chat.completions.create(
    model      = "gpt-4-1106-preview",
    messages   = messages,
    tools      = tools,
    tool_choice= "auto"
)

In [144]:
tool_call2 = response2.choices[0].message.tool_calls[0]
args2      = json.loads(tool_call2.function.arguments)

print("Function Arguments from Turn 2:")
print(args2)

Function Arguments from Turn 2:
{'age': 70, 'gender': 'female', 'symptoms': ['headache', 'fever', 'nausea'], 'duration_days': 3}


**4: Validate & Use with Pydantic**

In [145]:
try:
    patient = PatientInfo(**args2)
    
    print("✅ Final Patient Info:")
    print(patient.model_dump())
except Exception as e:
    print("❌ Validation error:", e)


✅ Final Patient Info:
{'name': None, 'age': 70, 'gender': 'female', 'symptoms': ['headache', 'fever', 'nausea'], 'duration_days': 3}


In [146]:
messages

[{'role': 'system',
  'content': 'You are a medical assistant that captures patient records.'},
 {'role': 'user',
  'content': 'I have a 70-year-old female patient with headache.'},
 {'role': 'user', 'content': 'She also has fever and nausea since 3 days.'}]

In [129]:
messages.append({
    "role": "assistant",
    "tool_calls": [tool_call]
})

In [130]:
messages

[{'role': 'system',
  'content': 'You are a medical assistant that captures patient records.'},
 {'role': 'user',
  'content': 'I have a 70-year-old female patient with headache.'},
 {'role': 'user', 'content': 'She also has fever and nausea since 3 days.'},
 {'role': 'assistant',
  'tool_calls': [ChatCompletionMessageToolCall(id='call_B3G3h8BDvZHiLDpd7zpg7or1', function=Function(arguments='{\n  "name": null,\n  "age": 65,\n  "gender": "male",\n  "symptoms": ["chronic cough", "chest pain"],\n  "duration_days": 7\n}', name='register_patient'), type='function')]}]

**Define the Tool Implementation**

In [131]:
def generate_care_plan(name, age, gender, symptoms, duration_days):
    return f"""🩺 Care Plan for {name}:
- Age: {age}
- Gender: {gender}
- Reported Symptoms: {symptoms}
- Duration : {duration_days}
- Suggested Action: Schedule consultation, basic tests, and monitor vitals daily.
"""

In [132]:
try:
    patient = PatientInfo(**args2)
    result  = generate_care_plan(**patient.model_dump())
except ValidationError as e:
    result = f"❌ Validation Error:\n{e}"

In [133]:
# Now post the tool output
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": result
})

**What This Demonstrates**

- Assistant `builds up memory` of patient details across user turns.
- `Function calling` is used each time to return structured arguments.
- You’re manually managing `thread memory` by keeping messages list.
- You can serialize patient object or store it in your DB / file.

**Why Messages Are Not Saved in a Thread**

- With the Responses API (client.chat.completions.create), OpenAI does not store message history. 

You must:

- Build and maintain the full messages list locally
- Append user ↔ assistant ↔ tool messages yourself
- Pass the entire conversation history in every call

This gives you full control — ideal for:

- Custom memory management
- Stateless chat servers
- Integrating your own memory backends (e.g., Redis, PostgreSQL)

END OF THE EAMPLE

---
#### Same patient care solution

- Responses API
- function calling
- Pydantic 
- persist conversation state to PostgreSQL
---

**Overview of Workflow**

1. User asks a health query

2. You:

    - Store the message in DB (e.g., table messages)
    - Accumulate the chat history
    - Call openai.chat.completions.create() with full history

3. If function call is returned:

    - Extract arguments
    - Validate with Pydantic
    - Call your function (e.g., triage_patient)
    - Store tool output in DB
    - Send tool result to LLM (new message)

Repeat…

#### Database Setup

In [208]:
import psycopg2
import uuid

In [209]:
DB_PARAMS = {
    "host": "localhost",
    "port": 5432,
    "database": "postgres",  
    "user": "postgres",
    "password": "xyz123"
}

Create Tables (Run Once)

In [210]:
conn = psycopg2.connect(**DB_PARAMS)
cur  = conn.cursor()

In [213]:
cur.execute("""
CREATE TABLE IF NOT EXISTS messages (
    id UUID PRIMARY KEY,
    conversation_id UUID,
    role TEXT CHECK (role IN ('user', 'assistant', 'tool')),
    content TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS tool_calls (
    id UUID PRIMARY KEY,
    conversation_id UUID,
    function_name TEXT,
    arguments JSONB,
    output TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
""")

conn.commit()
cur.close()
conn.close()

InterfaceError: cursor already closed

#### Function Tool + Pydantic

In [220]:
from pydantic import BaseModel, Field, field_validator
from typing import List
from openai import OpenAI
import json
import pprint

In [145]:
client = OpenAI()

In [169]:
class PatientRequest(BaseModel):
    name: str
    age: int            = Field(..., ge=0, le=120)
    gender: str
    symptoms: List[str]
    duration_days: int  = Field(..., ge=0, le=30)

Function

In [170]:
def triage_patient(name, age, gender, symptoms, duration_days):
    if "fever" in symptoms and duration_days >= 3:
        return f"{name} should visit a doctor. Fever lasting {duration_days} days needs evaluation."
    return f"{name}'s symptoms appear mild. Monitor and rest."

#### Tool Schema (for LLM)

In [204]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "triage_patient",
            "description": "Check symptoms, age and suggest urgency level",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "age": {"type": "integer"},
                    "gender": {"type": "string"},
                    "symptoms": {
                        "type": "array",
                        "items": {"type": "string"}
                    },
                    "duration_days": {"type": "integer"}
                },
                "required": ["name", "age", "gender", "symptoms", "duration_days"]
            }
        }
    }
]

In [237]:
def load_history(conversation_id):
    with psycopg2.connect(**DB_PARAMS) as conn:
        with conn.cursor() as cur:
            cur.execute(
                "SELECT role, content FROM messages WHERE conversation_id = %s ORDER BY created_at  ASC",
                [str(conversation_id)]
            )
            rows = cur.fetchall()
            messages = []
            for role, content in rows:
                if role == "tool":
                    data = json.loads(content)
                    messages.append({
                        "role": "tool",
                        "tool_call_id": data["tool_call_id"],
                        "content": data["content"]
                    })
                else:
                    try:
                        content_data = json.loads(content)
                        if "tool_calls" in content_data:
                            messages.append({
                                "role": "assistant",
                                "content": content_data.get("content"),
                                "tool_calls": content_data["tool_calls"]
                            })
                        else:
                            messages.append({"role": role, "content": content})
                    except json.JSONDecodeError:
                        messages.append({"role": role, "content": content})
            return messages


#### Run First Message

In [221]:
conversation_id = uuid.uuid4()

In [222]:
pprint.pprint(load_history(conversation_id))

[]


In [214]:
def save_message(conversation_id, role, content, tool_call_id=None):
    with psycopg2.connect(**DB_PARAMS) as conn:
        with conn.cursor() as cur:
            if role == "tool":
                payload = json.dumps({
                    "tool_call_id": tool_call_id,
                    "content": content
                })
                cur.execute(
                    "INSERT INTO messages (id, conversation_id, role, content) VALUES (%s, %s, %s, %s)",
                    [str(uuid.uuid4()), str(conversation_id), role, payload]
                )
            else:
                cur.execute(
                    "INSERT INTO messages (id, conversation_id, role, content) VALUES (%s, %s, %s, %s)",
                    [str(uuid.uuid4()), str(conversation_id), role, content]
                )
            conn.commit()

In [223]:
# Turn 1: User input
user_msg1 = "My mom is 70 years old, has headache and fever since 3 days."
save_message(conversation_id, "user", user_msg1)

# Step 1: Call LLM
response1 = client.chat.completions.create(
    model       = "gpt-4-1106-preview",
    messages    = load_history(conversation_id),
    tools       = tools,
    tool_choice = "auto"
)

In [227]:
assistant_msg1 = response1.choices[0].message
assistant_msg1

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_emdvWMzDZjWFsvdUbXIWNFGR', function=Function(arguments='{"name":"mom","age":70,"gender":"female","symptoms":["headache","fever"],"duration_days":3}', name='triage_patient'), type='function')])

In [228]:
save_message(conversation_id, "assistant", assistant_msg1.model_dump_json())

In [229]:
pprint.pprint(load_history(conversation_id))

[{'content': 'My mom is 70 years old, has headache and fever since 3 days.',
  'role': 'user'},
 {'content': None,
  'role': 'assistant',
  'tool_calls': [{'function': {'arguments': '{"name":"mom","age":70,"gender":"female","symptoms":["headache","fever"],"duration_days":3}',
                               'name': 'triage_patient'},
                  'id': 'call_emdvWMzDZjWFsvdUbXIWNFGR',
                  'type': 'function'}]}]


#### Call Function and Store Output

In [230]:
tool_call1 = assistant_msg1.tool_calls[0]
tool_call1

ChatCompletionMessageToolCall(id='call_emdvWMzDZjWFsvdUbXIWNFGR', function=Function(arguments='{"name":"mom","age":70,"gender":"female","symptoms":["headache","fever"],"duration_days":3}', name='triage_patient'), type='function')

In [231]:
args = json.loads(tool_call1.function.arguments)
args

{'name': 'mom',
 'age': 70,
 'gender': 'female',
 'symptoms': ['headache', 'fever'],
 'duration_days': 3}

In [232]:
result = triage_patient(**args)
result

'mom should visit a doctor. Fever lasting 3 days needs evaluation.'

In [233]:
save_message(conversation_id, "tool", result, tool_call_id=tool_call1.id)

In [234]:
pprint.pprint(load_history(conversation_id))

[{'content': 'My mom is 70 years old, has headache and fever since 3 days.',
  'role': 'user'},
 {'content': None,
  'role': 'assistant',
  'tool_calls': [{'function': {'arguments': '{"name":"mom","age":70,"gender":"female","symptoms":["headache","fever"],"duration_days":3}',
                               'name': 'triage_patient'},
                  'id': 'call_emdvWMzDZjWFsvdUbXIWNFGR',
                  'type': 'function'}]},
 {'content': 'mom should visit a doctor. Fever lasting 3 days needs '
             'evaluation.',
  'role': 'tool',
  'tool_call_id': 'call_emdvWMzDZjWFsvdUbXIWNFGR'}]


#### Turn 2 - New User Message

In [235]:
user_msg2 = "Now she also has nausea and dizziness."
save_message(conversation_id, "user", user_msg2)


In [238]:
pprint.pprint(load_history(conversation_id))

[{'content': 'My mom is 70 years old, has headache and fever since 3 days.',
  'role': 'user'},
 {'content': None,
  'role': 'assistant',
  'tool_calls': [{'function': {'arguments': '{"name":"mom","age":70,"gender":"female","symptoms":["headache","fever"],"duration_days":3}',
                               'name': 'triage_patient'},
                  'id': 'call_emdvWMzDZjWFsvdUbXIWNFGR',
                  'type': 'function'}]},
 {'content': 'mom should visit a doctor. Fever lasting 3 days needs '
             'evaluation.',
  'role': 'tool',
  'tool_call_id': 'call_emdvWMzDZjWFsvdUbXIWNFGR'},
 {'content': 'Now she also has nausea and dizziness.', 'role': 'user'}]


**Call the LLM again with full memory**

In [239]:
response2 = client.chat.completions.create(
    model       = "gpt-4-1106-preview",
    messages    = load_history(conversation_id),
    tools       = tools,
    tool_choice = "auto"
)

**Extract tool call from LLM reply**

In [240]:
assistant_msg2 = response2.choices[0].message
tool_call2 = assistant_msg2.tool_calls[0]

save_message(conversation_id, "assistant", "")  # tool_call present, so content is None

In [241]:
pprint.pprint(load_history(conversation_id))

[{'content': 'My mom is 70 years old, has headache and fever since 3 days.',
  'role': 'user'},
 {'content': None,
  'role': 'assistant',
  'tool_calls': [{'function': {'arguments': '{"name":"mom","age":70,"gender":"female","symptoms":["headache","fever"],"duration_days":3}',
                               'name': 'triage_patient'},
                  'id': 'call_emdvWMzDZjWFsvdUbXIWNFGR',
                  'type': 'function'}]},
 {'content': 'mom should visit a doctor. Fever lasting 3 days needs '
             'evaluation.',
  'role': 'tool',
  'tool_call_id': 'call_emdvWMzDZjWFsvdUbXIWNFGR'},
 {'content': 'Now she also has nausea and dizziness.', 'role': 'user'},
 {'content': '', 'role': 'assistant'}]


**Parse arguments + Call your function**

In [242]:
args2 = json.loads(tool_call2.function.arguments)
result2 = triage_patient(**args2)

**Save tool result**

In [243]:
save_message(conversation_id, "tool", result2, tool_call_id=tool_call2.id)


**now inspect the full conversation memory again**

In [244]:
pprint.pprint(load_history(conversation_id))

[{'content': 'My mom is 70 years old, has headache and fever since 3 days.',
  'role': 'user'},
 {'content': None,
  'role': 'assistant',
  'tool_calls': [{'function': {'arguments': '{"name":"mom","age":70,"gender":"female","symptoms":["headache","fever"],"duration_days":3}',
                               'name': 'triage_patient'},
                  'id': 'call_emdvWMzDZjWFsvdUbXIWNFGR',
                  'type': 'function'}]},
 {'content': 'mom should visit a doctor. Fever lasting 3 days needs '
             'evaluation.',
  'role': 'tool',
  'tool_call_id': 'call_emdvWMzDZjWFsvdUbXIWNFGR'},
 {'content': 'Now she also has nausea and dizziness.', 'role': 'user'},
 {'content': '', 'role': 'assistant'},
 {'content': 'mom should visit a doctor. Fever lasting 3 days needs '
             'evaluation.',
  'role': 'tool',
  'tool_call_id': 'call_PBfexN0eEKMHAe2qpykzikb5'}]
